# 04 Match rate: Gate 1 (T1.3)

**Task:** T1.3. Measures how well automatic UEI/DUNS matching through the harbor–entity crosswalk links awards to the 20 pilot harbors.

**Denominator:** a hand-coded truth set (`data/reference/award_harbor_truth_pilot.csv`), not a share of all in-scope awards. The settled definition (decisions log, 2026-11-18) is:
- **Recall:** the share of awards at a sample harbor, by count and by dollars, that matching assigns to the right harbor with no manual step.
- **Precision:** the share of auto-assigned awards that land on the right harbor.

**Reads:**
- `data/processed/usaspending_pilot/{prime_awards,subawards}.csv` (T1.1)
- `data/processed/bric_pacific_subapplicants.csv` (T2.5a)
- `data/reference/harbor_entity_crosswalk.csv` (T1.2)
- the truth, override and subapplicant-review tables (T1.3)

**Writes:** `data/processed/match_pilot/` (not committed).

**Matching rule (`harborfund.link.crosswalk_match`):**
1. UEI.
2. Legacy DUNS, only if the UEI misses.
3. Location check: the matched harbor's county must equal the award's recipient county or place-of-performance county.

A matched award with a location disagreement, or matching several harbors, is **flagged** for review, not auto-assigned. Dollars are nominal.

In [1]:
import pandas as pd
from harborfund import config
from harborfund.link import crosswalk_match as cm

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
summary = cm.run()          # builds universe + matches for every crosswalk version
summary

{'sample_counties': 17,
 'review_universe': {'n': 241,
  'usd_nominal': 782190178.71,
  'by_source': {'prime': 183, 'subaward': 58},
  'dropped_duplicate_subaward_reports': 0},
 'subapplicant_universe': {'n': 441},
 'match_status': {'own_op': {'none': 218, 'auto': 22, 'flagged_geo': 1},
  'own_op_nolow': {'none': 218, 'auto': 22, 'flagged_geo': 1},
  'own_op_parent': {'none': 217, 'auto': 23, 'flagged_geo': 1},
  'own_op_parent_nolow': {'none': 218, 'auto': 22, 'flagged_geo': 1}}}

## 1. Review universe

In [2]:
OUT = cm.OUT_DIR
uni = pd.read_csv(OUT / "review_universe.csv", dtype={"award_id": str})
truth = pd.read_csv(cm.TRUTH_PATH, dtype=str, keep_default_na=False)
truth["amount_nominal"] = pd.to_numeric(truth["amount_nominal"])
assert len(uni) == len(truth) and set(zip(uni.award_id, uni.award_source)) == set(zip(truth.award_id, truth.award_source))

tab = (uni.groupby(["award_source", "program_id"])
          .agg(n=("award_id", "size"), usd_m=("amount_nominal", lambda s: s.sum() / 1e6),
               kw_flag=("harbor_kw_flag", "sum"))
          .round(1))
tab.loc[("all", ""), :] = [len(uni), round(uni.amount_nominal.sum() / 1e6, 1), uni.harbor_kw_flag.sum()]
tab

n  usd_m  kw_flag
award_source program_id                           
prime        bric               22    4.0        0
             eda_eaa           107  213.9       10
             eda_public_works   21   48.9        5
             other_20823         1    5.6        0
             other_97047         1    0.2        0
             pdm_competitive    10    3.2        0
             pdm_cpf             1    1.4        0
             pidp               20  336.4       20
subaward     bric               22   70.1        0
             eda_eaa            14   14.6        0
             pdm_competitive    10    9.5        0
             pdm_cpf             8   21.7        0
             pidp                4   52.9        4
all                            241  782.2       39

## 2. Truth labels

In [3]:
pd.crosstab([truth.award_source, truth.program_id], truth.label, margins=True)

label                          not_harbor  other_harbor  sample_harbor  unclear  All
award_source program_id                                                             
prime        bric                      21             0              0        1   22
             eda_eaa                  102             4              1        0  107
             eda_public_works          15             4              2        0   21
             other_20823                0             0              1        0    1
             other_97047                1             0              0        0    1
             pdm_competitive           10             0              0        0   10
             pdm_cpf                    1             0              0        0    1
             pidp                       0             9             11        0   20
subaward     bric                      22             0              0        0   22
             eda_eaa                   14             0              0        0   14
             pdm_competitive           10             0              0        0   10
             pdm_cpf                    8             0              0        0    8
             pidp                       0             3              1        0    4
All                                   204            20             16        1  241

In [4]:
# who the 16 sample-harbor awards are
truth.query("label == 'sample_harbor'")[["award_id", "award_source", "program_id", "harbor_id",
                                         "recipient_name", "amount_nominal"]].sort_values("harbor_id")

,award_id,award_source,program_id,harbor_id,recipient_name,amount_nominal
169,693JF72344010,prime,pidp,CA-crescent-city,CRESCENT CITY HARBOR DISTRICT,7366566.0
170,693JF72540013,prime,pidp,CA-crescent-city,CRESCENT CITY HARBOR DISTRICT,8000000.0
163,693JF72344005,prime,pidp,CA-humboldt-bay,CITY OF EUREKA,650000.0
178,693JF72444090,prime,pidp,CA-humboldt-bay,PORT OF GRAYS HARBOR,8672986.0
129,070107315,prime,eda_public_works,CA-los-angeles-long-beach,ALTASEA AT THE PORT OF LOS ANGELES,3000000.0
164,693JF72040020,prime,pidp,CA-los-angeles-long-beach,CITY OF LONG BEACH,14500000.0
165,693JF722450027,prime,pidp,CA-los-angeles-long-beach,CITY OF LONG BEACH,52300000.0
166,693JF72344024,prime,pidp,CA-los-angeles-long-beach,CITY OF LONG BEACH,30141080.0
167,693JF72444055,prime,pidp,CA-los-angeles-long-beach,CITY OF LONG BEACH,52633331.0
171,693JF72040021,prime,pidp,CA-los-angeles-long-beach,"LOS ANGELES, CITY OF",18184743.0


In [5]:
truth.groupby("label_method").size().rename("awards")

label_method
desc    207
fema     29
web       5
Name: awards, dtype: int64

## 3. Recall and precision: every version

Versions follow kickoff step 4:
- owner/operator rows only, or plus `parent_government` rows;
- each with and without `low`-confidence rows.

**geo gate:** the primary measure. An ID match also needs the location check to pass.
**ID only:** the assignment the UEI/DUNS match alone would give.

**site / owner:** the truth view (see the truth-set notes).
- `site` (primary) counts only projects at the named harbor.
- `owner` also credits a pilot owner's other harbor-related facilities (3 awards) to that owner's harbor.

In [6]:
matches = pd.read_csv(OUT / "award_matches.csv", dtype={"award_id": str})
matches["amount_nominal"] = pd.to_numeric(matches["amount_nominal"])
truth_views = {v: cm.apply_truth_view(truth, v) for v in ("site", "owner")}

rows = []
for view, t in truth_views.items():
    for version in cm.VERSIONS:
        mv = matches[matches.version == version]
        for gate in (True, False):
            s = cm.score(mv, t, gate_geo=gate)
            rows.append({"truth_view": view, "version": version, "match": "geo gate" if gate else "ID only", **s})
res = pd.DataFrame(rows)
show = ["truth_view", "version", "match", "n_truth_sample", "n_correct", "recall_n", "recall_usd",
        "n_assigned", "precision_n", "precision_usd", "n_unclear"]
res[show].round(3)

,truth_view,version,match,n_truth_sample,n_correct,recall_n,recall_usd,n_assigned,precision_n,precision_usd,n_unclear
0,site,own_op,geo gate,16.0,13.0,0.812,0.839,22.0,0.591,0.833,1.0
1,site,own_op,ID only,16.0,13.0,0.812,0.839,23.0,0.565,0.806,1.0
2,site,own_op_nolow,geo gate,16.0,13.0,0.812,0.839,22.0,0.591,0.833,1.0
3,site,own_op_nolow,ID only,16.0,13.0,0.812,0.839,23.0,0.565,0.806,1.0
4,site,own_op_parent,geo gate,16.0,13.0,0.812,0.839,23.0,0.565,0.805,1.0
5,site,own_op_parent,ID only,16.0,13.0,0.812,0.839,24.0,0.542,0.780,1.0
6,site,own_op_parent_nolow,geo gate,16.0,13.0,0.812,0.839,22.0,0.591,0.833,1.0
7,site,own_op_parent_nolow,ID only,16.0,13.0,0.812,0.839,23.0,0.565,0.806,1.0
8,owner,own_op,geo gate,19.0,16.0,0.842,0.860,22.0,0.727,0.981,1.0
9,owner,own_op,ID only,19.0,16.0,0.842,0.860,23.0,0.696,0.949,1.0


**Pass-through subawards.** The LBCT LLC subaward ($30.1M) repeats the dollars of POLB's PIDP prime 693JF72344024, so summing primes and subawards counts that money twice. Below, recall is shown with pass-through subawards (whose prime is also in the universe) dropped. This is the dollar view T2.2 should use.

In [7]:
pt = truth.passthrough_of_prime != ""
rows = []
for view, t in truth_views.items():
    t2 = t[~pt.values]
    for version in ("own_op", "own_op_parent"):
        mv = matches[(matches.version == version)].merge(t2[["award_id", "award_source"]])
        s = cm.score(mv, t2)
        rows.append({"truth_view": view, "version": version, **s})
pd.DataFrame(rows)[["truth_view", "version", "n_truth_sample", "n_correct", "recall_n", "recall_usd",
                    "n_assigned", "precision_n", "precision_usd"]].round(3)

,truth_view,version,n_truth_sample,n_correct,recall_n,recall_usd,n_assigned,precision_n,precision_usd
0,site,own_op,15.0,13.0,0.867,0.949,22.0,0.591,0.833
1,site,own_op_parent,15.0,13.0,0.867,0.949,23.0,0.565,0.805
2,owner,own_op,18.0,16.0,0.889,0.956,22.0,0.727,0.981
3,owner,own_op_parent,18.0,16.0,0.889,0.956,23.0,0.696,0.948


In [8]:
# Small-sample uncertainty: Wilson 95% interval for count recall (primary: site view, own_op_parent, geo gate)
from math import sqrt
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return round(c - h, 3), round(c + h, 3)
for label, k, n in [("with pass-through", 13, 16), ("without pass-through", 13, 15)]:
    print(f"recall {k}/{n} = {k/n:.3f}, 95% CI {wilson(k, n)}  ({label})")
print("precision 13/23 =", round(13/23, 3), "95% CI", wilson(13, 23))

recall 13/16 = 0.812, 95% CI (0.57, 0.934)  (with pass-through)
recall 13/15 = 0.867, 95% CI (0.621, 0.963)  (without pass-through)
precision 13/23 = 0.565 95% CI (0.368, 0.744)


## 4. By program and by owner type (primary: site view, `own_op_parent`, geo gate)

In [9]:
prim = matches[matches.version == cm.PRIMARY_VERSION].copy()
t_site = truth_views["site"]
cols = ["n_truth_sample", "usd_truth_sample", "n_correct", "recall_n", "recall_usd", "n_assigned", "precision_n", "precision_usd"]
by_prog = cm.score_by(prim, t_site, "program_id")[cols]
by_prog.loc["bric (subawards only)"] = cm.score(prim[(prim.program_id == "bric") & (prim.award_source == "subaward")],
                                                 t_site.merge(prim.loc[(prim.program_id == "bric") & (prim.award_source == "subaward"), ["award_id", "award_source"]]))[cols]
by_prog.round(3)

,n_truth_sample,usd_truth_sample,n_correct,recall_n,recall_usd,n_assigned,precision_n,precision_usd
bric,0.0,0.0,0.0,NaN,NaN,2.0,0.000,0.000
eda_eaa,1.0,6200000.0,1.0,1.000,1.000,4.0,0.250,0.330
eda_public_works,2.0,4195563.0,1.0,0.500,0.285,1.0,1.000,1.000
other_20823,1.0,5550000.0,1.0,1.000,1.000,1.0,1.000,1.000
other_97047,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
pdm_competitive,0.0,0.0,0.0,NaN,NaN,2.0,0.000,0.000
pdm_cpf,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN
pidp,12.0,243719786.0,10.0,0.833,0.841,13.0,0.769,0.841
bric (subawards only),0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN


In [10]:
# owner type: of the true harbor for recall; of the assigned harbor for precision
x = prim.merge(t_site[["award_id", "award_source", "label", "harbor_id"]], on=["award_id", "award_source"])
x["true_owner_type"] = x.harbor_id.map(cm.OWNER_TYPE)
x["assigned_owner_type"] = x.auto_harbor_id.map(cm.OWNER_TYPE)
rec = (x[x.label == "sample_harbor"].assign(ok=lambda d: d.auto_harbor_id == d.harbor_id)
         .groupby("true_owner_type").agg(n_truth=("ok", "size"), n_correct=("ok", "sum"),
                                          usd_truth=("amount_nominal", "sum"),
                                          usd_correct=("amount_nominal", lambda s: s[x.loc[s.index, "auto_harbor_id"] == x.loc[s.index, "harbor_id"]].sum())))
rec["recall_n"] = rec.n_correct / rec.n_truth
rec["recall_usd"] = rec.usd_correct / rec.usd_truth
a = x[x.auto_harbor_id.notna() & (x.label != "unclear")]
prec = a.assign(ok=a.auto_harbor_id == a.harbor_id).groupby("assigned_owner_type").agg(n_assigned=("ok", "size"), n_correct_assigned=("ok", "sum"))
prec["precision_n"] = prec.n_correct_assigned / prec.n_assigned
pd.concat([rec, prec], axis=1).round(3)

,n_truth,n_correct,usd_truth,usd_correct,recall_n,recall_usd,n_assigned,n_correct_assigned,precision_n
city (incl. port dept),8.0,6.0,210780234.0,177639154.0,0.75,0.843,8,6,0.75
mixed/private,2.0,1.0,9322986.0,650000.0,0.50,0.070,1,1,1.00
port/harbor district,6.0,6.0,39562129.0,39562129.0,1.00,1.000,10,6,0.60
tribal,NaN,NaN,NaN,NaN,NaN,NaN,4,0,0.00


In [11]:
# harbors: which pilot harbors have any sample-harbor award at all
h = pd.DataFrame({"owner_type": pd.Series(cm.OWNER_TYPE)})
h["truth_awards"] = t_site.query("label == 'sample_harbor'").harbor_id.value_counts()
h["auto_correct"] = x[(x.label == "sample_harbor") & (x.auto_harbor_id == x.harbor_id)].harbor_id.value_counts()
h["auto_assigned_total"] = prim.auto_harbor_id.value_counts()
h.fillna(0).astype({"truth_awards": int, "auto_correct": int, "auto_assigned_total": int}).sort_values("truth_awards", ascending=False)

,owner_type,truth_awards,auto_correct,auto_assigned_total
CA-los-angeles-long-beach,city (incl. port dept),8,6,8
CA-humboldt-bay,mixed/private,2,1,1
CA-san-diego-bay,port/harbor district,2,2,2
CA-crescent-city,port/harbor district,2,2,2
WA-westport,port/harbor district,1,1,2
OR-newport,port/harbor district,1,1,2
CA-trinidad,tribal,0,0,3
CA-santa-cruz,port/harbor district,0,0,0
CA-channel-islands,county,0,0,0
CA-dana-point,county,0,0,0


### PIDP earmark-candidate years (FY2024+), reported separately as the kickoff asks

In [12]:
e = prim[prim.earmark_candidate_year.astype(str) == "True"]
te = t_site.merge(e[["award_id", "award_source"]])
pd.DataFrame({"earmark_candidate_year (FY2024+)": cm.score(e, te),
              "other years": cm.score(prim[prim.earmark_candidate_year.astype(str) != "True"],
                                      t_site.merge(prim.loc[prim.earmark_candidate_year.astype(str) != "True", ["award_id", "award_source"]]))}).T.round(3)

,n_truth_sample,usd_truth_sample,n_correct,usd_correct,recall_n,recall_usd,n_assigned,usd_assigned,precision_n,precision_usd,n_unclear
earmark_candidate_year (FY2024+),10.0,206705043.0,8.0,167890977.0,0.800,0.812,11.0,1.969776e+08,0.727,0.852,0.0
other years,6.0,52960306.0,5.0,49960306.0,0.833,0.943,12.0,7.362895e+07,0.417,0.679,1.0


## 5. Error review: every miss and every misassignment

In [13]:
CAUSE = {
    "693JF72444090": ("wrong UEI", "Port of Grays Harbor's UEI on a Humboldt Bay award; location check flagged it, so a review step (not a misassignment)"),
    "070107315": ("entity not in crosswalk", "tenant nonprofit (AltaSea) on Port of LA property"),
    "077907489": ("parent government received it", "City of Long Beach general UEI; citywide flood control, not port"),
    "077907579": ("other: owner UEI shared with city programs", "Harbor Department UEI also carries the City's CARES Act revolving loan fund"),
    "077907707": ("other: owner's non-harbor award", "Port of Skagit business park, not La Conner Marina"),
    "EMF-2020-PC-0003": ("other: owner's non-harbor award", "tribal owner's hazard-mitigation project"),
    "EMF-2020-PC-0009": ("other: owner's non-harbor award", "tribal owner's generator / plan"),
    "EMF-2022-BR-020": ("other: owner's non-harbor award", "tribal owner's plan and stabilization"),
    "EMS-2020-BR-127-0001": ("other: owner's non-harbor award", "tribal owner's hazard mitigation plan"),
    "693JF72140011": ("other: owner's other facility (site rule)", "Coos Bay Rail Line, not Charleston Marina"),
    "693JF72344028": ("other: owner's other facility (site rule)", "Grays Harbor Terminal 4 (Aberdeen), not Westport Marina"),
    "693JF72444084": ("other: owner's other facility (site rule)", "Newport International Terminal, not the commercial marina"),
}
x2 = matches[matches.version == cm.PRIMARY_VERSION].merge(
    t_site[["award_id", "award_source", "label", "harbor_id", "scope"]], on=["award_id", "award_source"])
miss = x2[(x2.label == "sample_harbor") & (x2.auto_harbor_id != x2.harbor_id)].assign(error="miss")
mis = x2[x2.auto_harbor_id.notna() & (x2.auto_harbor_id != x2.harbor_id) & (x2.label != "unclear")].assign(error="misassignment")
err = pd.concat([miss, mis])
key = err.award_id.str.split("|").str[0]
err["cause"] = [CAUSE.get(k, ("entity not in crosswalk", "pass-through subrecipient (terminal operator)"))[0] for k in key]
err["detail"] = [CAUSE.get(k, ("", "LBCT LLC, pass-through of POLB PIDP 693JF72344024"))[1] for k in key]
err_tab = err[["error", "award_id", "award_source", "program_id", "recipient_name", "amount_nominal", "match_status",
               "id_harbor_id", "harbor_id", "cause", "detail"]].rename(columns={"id_harbor_id": "id_matched_harbor", "harbor_id": "true_harbor"})
err_tab.to_csv(OUT / "error_table.csv", index=False)
err_tab

,error,award_id,award_source,program_id,recipient_name,amount_nominal,match_status,id_matched_harbor,true_harbor,cause,detail
15,miss,693JF72444090,prime,pidp,PORT OF GRAYS HARBOR,8672986.00,flagged_geo,WA-westport,CA-humboldt-bay,wrong UEI,Port of Grays Harbor's UEI on a Humboldt Bay award; location check flagged i...
31,miss,070107315,prime,eda_public_works,ALTASEA AT THE PORT OF LOS ANGELES,3000000.00,none,NaN,CA-los-angeles-long-beach,entity not in crosswalk,tenant nonprofit (AltaSea) on Port of LA property
185,miss,693JF72344024|MARAD PIDP 2022 HD-24-0122|E3E32FA8-4EA3-4202-A949-8540C3BCEC06,subaward,pidp,LBCT LLC,30141080.00,none,NaN,CA-los-angeles-long-beach,entity not in crosswalk,"LBCT LLC, pass-through of POLB PIDP 693JF72344024"
3,misassignment,693JF72140011,prime,pidp,OREGON INTERNATIONAL PORT OF COOS BAY,9869031.60,auto,OR-charleston,,other: owner's other facility (site rule),"Coos Bay Rail Line, not Charleston Marina"
10,misassignment,693JF72344028,prime,pidp,PORT OF GRAYS HARBOR,25500000.00,auto,WA-westport,,other: owner's other facility (site rule),"Grays Harbor Terminal 4 (Aberdeen), not Westport Marina"
13,misassignment,693JF72444084,prime,pidp,PORT OF NEWPORT,3444100.00,auto,OR-newport,,other: owner's other facility (site rule),"Newport International Terminal, not the commercial marina"
89,misassignment,077907489,prime,eda_eaa,CITY OF LONG BEACH,8960000.00,auto,CA-los-angeles-long-beach,,parent government received it,"City of Long Beach general UEI; citywide flood control, not port"
100,misassignment,077907579,prime,eda_eaa,CITY OF LONG BEACH,1402449.79,auto,CA-los-angeles-long-beach,,other: owner UEI shared with city programs,Harbor Department UEI also carries the City's CARES Act revolving loan fund
116,misassignment,077907707,prime,eda_eaa,PORT OF SKAGIT COUNTY,2198400.00,auto,WA-la-conner,,other: owner's non-harbor award,"Port of Skagit business park, not La Conner Marina"
151,misassignment,EMF-2020-PC-0003,prime,pdm_competitive,CHER-AE HEIGHTS INDIAN COMMUNITY OF THE TRINIDAD RANCHERIA,594576.00,auto,CA-trinidad,,other: owner's non-harbor award,tribal owner's hazard-mitigation project


In [14]:
err_tab.groupby(["error", "cause"]).agg(n=("award_id", "size"), usd_m=("amount_nominal", lambda s: round(s.sum() / 1e6, 2)))

n  usd_m
error         cause                                               
misassignment other: owner UEI shared with city programs  1   1.40
              other: owner's non-harbor award             5   3.58
              other: owner's other facility (site rule)   3  38.81
              parent government received it               1   8.96
miss          entity not in crosswalk                     2  33.14
              wrong UEI                                   1   8.67

What each cause implies for T2.1 and T2.2:

| Cause | Pilot cases | Implication |
|---|---|---|
| Wrong UEI | 1 ($8.7M) | Keep the location check as a gate. It turned a misassignment into a review flag. Review flagged awards by hand |
| Entity not in crosswalk | 2 ($33.1M, $30.1M of it a pass-through) | T2.1 adds tenants and terminal operators only where they hold harbor awards. T2.2 drops pass-through subawards whose prime is already counted, or the dollars double |
| Parent government received it | 1 ($9.0M, not harbor) | `parent_government` rows add no correct match and one false one. Leave them out of auto-matching, or require a harbor keyword |
| Owner UEI shared with city programs | 1 ($1.4M) | Big-city "Harbor Department" UEIs aren't port-only. Entity match needs a harbor-relatedness check |
| Owner's non-harbor award | 5 ($3.6M) | Tribal, county and port owners get hazard-mitigation and business-park money. Entity match alone over-assigns; add a harbor-relatedness filter (section 6) |
| Owner's other facility (site rule) | 3 ($38.8M) | Multi-facility ports (Coos Bay, Grays Harbor, Newport). T2.1 must decide site vs owner attribution and, under the site rule, add a facility field |

## 6. Keyword flag (`harbor_kw_flag`) as a pre-filter for T2.2

The question: does the flag separate harbor-related awards (sample + other harbor) from the rest? For PIDP the flag is always on (port program); for other programs it is a word-start keyword match on the description.

In [15]:
kw = uni.merge(truth[["award_id", "award_source", "label"]], on=["award_id", "award_source"])
kw = kw[kw.label != "unclear"]
kw["harbor_related"] = kw.label.isin(["sample_harbor", "other_harbor"])
def kw_stats(d):
    tp = (d.harbor_kw_flag & d.harbor_related).sum(); fp = (d.harbor_kw_flag & ~d.harbor_related).sum()
    fn = (~d.harbor_kw_flag & d.harbor_related).sum(); tn = (~d.harbor_kw_flag & ~d.harbor_related).sum()
    usd_rel = d.loc[d.harbor_related, "amount_nominal"].sum()
    return pd.Series({"harbor_related": tp + fn, "flagged": tp + fp, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                      "sensitivity": tp / (tp + fn) if tp + fn else float("nan"),
                      "sensitivity_usd": d.loc[d.harbor_kw_flag & d.harbor_related, "amount_nominal"].sum() / usd_rel if usd_rel else float("nan"),
                      "precision": tp / (tp + fp) if tp + fp else float("nan"),
                      "share_flagged": (tp + fp) / len(d)})
kw_all = kw_stats(kw).rename("all")
kw_nonpidp = kw_stats(kw[kw.program_id != "pidp"]).rename("excluding PIDP (flag always on)")
kw_by = kw.groupby("program_id").apply(kw_stats, include_groups=False)
pd.concat([kw_all.to_frame().T, kw_nonpidp.to_frame().T, kw_by]).round(3)

,harbor_related,flagged,tp,fp,fn,tn,sensitivity,sensitivity_usd,precision,share_flagged
all,36.0,39.0,33.0,6.0,3.0,198.0,0.917,0.980,0.846,0.162
excluding PIDP (flag always on),12.0,15.0,9.0,6.0,3.0,198.0,0.750,0.765,0.600,0.069
bric,0.0,0.0,0.0,0.0,0.0,43.0,NaN,NaN,NaN,0.000
eda_eaa,5.0,10.0,4.0,6.0,1.0,110.0,0.800,0.959,0.400,0.083
eda_public_works,6.0,5.0,5.0,0.0,1.0,15.0,0.833,0.785,1.000,0.238
other_20823,1.0,0.0,0.0,0.0,1.0,0.0,0.000,0.000,NaN,0.000
other_97047,0.0,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,0.000
pdm_competitive,0.0,0.0,0.0,0.0,0.0,20.0,NaN,NaN,NaN,0.000
pdm_cpf,0.0,0.0,0.0,0.0,0.0,9.0,NaN,NaN,NaN,0.000
pidp,24.0,24.0,24.0,0.0,0.0,0.0,1.000,1.000,1.000,1.000


In [16]:
# harbor-related awards the flag misses, and non-harbor awards it catches
kw[(kw.harbor_related & ~kw.harbor_kw_flag) | (~kw.harbor_related & kw.harbor_kw_flag & (kw.program_id != "pidp"))][
    ["award_id", "program_id", "recipient_name", "label", "harbor_kw_flag", "description"]].assign(
    description=lambda d: d.description.str[:90])

,award_id,program_id,recipient_name,label,harbor_kw_flag,description
6,693JF72340010,other_20823,SAN DIEGO UNIFIED PORT DISTRICT,sample_harbor,False,AWARD PURPOSE THE PROJECT WILL FUND INFRASTRUCTURE UPGRADES THAT WILL BE DEP...
29,070107214,eda_public_works,PORT ANGELES PORT OF,other_harbor,False,FACILITY RENOVATION
70,077906895,eda_eaa,CITY OF SANTA CRUZ,other_harbor,False,DESIGN MASTER PLAN
116,077907707,eda_eaa,PORT OF SKAGIT COUNTY,not_harbor,True,THIS EDA INVESTMENT SUPPORTS THE PORT OF SKAGIT COUNTY WITH MAKING UTILITY I...
124,077907849,eda_eaa,CITY OF LONG BEACH,not_harbor,True,THIS EDA INVESTMENT SUPPORTS THE CITY OF LONG BEACH WITH BOOSTING BUSINESS A...
125,077907862,eda_eaa,PORT ANGELES WATERFRONT CENTER,not_harbor,True,THIS EDA INVESTMENT SUPPORTS THE PORT ANGELES WATERFRONT CENTER WITH BOOSTIN...
135,ED22HDQ3070010,eda_eaa,ECONOMIC DEVELOPMENT CORPORATION OF LOS ANGELES COUNTY,not_harbor,True,"THIS EDA INVESTMENT SUPPORTS THE COALITION, LED BY THE LOS ANGELES COUNTY EC..."
142,ED24SEA0G0017,eda_eaa,LOWER ELWHA TRIBAL COMMUNITY COUNCIL,not_harbor,True,THIS EDA INVESTMENT FUNDS THE LOWER ELWHA WASTE WATER INFRASTRUCTURE FEASIBI...
146,ED25SEA0G0021,eda_eaa,CLALLAM COUNTY ECONOMIC DEVELOPMENT COUNCIL,not_harbor,True,PURPOSE: THE PURPOSE OF THIS FINANCIAL ASSISTANCE AWARD IS TO SUPPORT THE IM...


**Combining entity match and keyword.** The proposed T2.2 workflow:
1. Auto-assign entity matches that pass the location check **and** carry the keyword flag.
2. Send to manual review every entity match without the flag, every location flag, and every keyword-flagged award with no entity match.

In [17]:
y = prim.merge(t_site[["award_id", "award_source", "label", "harbor_id"]], on=["award_id", "award_source"])
y["auto_kw"] = y.auto_harbor_id.where(y.harbor_kw_flag.astype(str) == "True")
s_kw = cm.score(y.assign(auto_harbor_id=y.auto_kw), t_site)
s_plain = cm.score(prim, t_site)
review = (y.id_harbor_id.notna() & y.auto_kw.isna()) | ((y.harbor_kw_flag.astype(str) == "True") & y.id_harbor_id.isna())
caught = y.label.eq("sample_harbor") & (y.auto_kw.eq(y.harbor_id) | review)
out = pd.DataFrame({"entity match (geo gate)": s_plain, "entity match + keyword": s_kw}).loc[
    ["n_correct", "recall_n", "recall_usd", "n_assigned", "precision_n", "precision_usd"]]
print(out.round(3).to_string())
print(f"\nmanual review queue: {review.sum()} of {len(y)} awards ({review.mean():.1%}); "
      f"sample-harbor awards caught by auto or review: {caught.sum()} of {y.label.eq('sample_harbor').sum()}")

               entity match (geo gate)  entity match + keyword
n_correct                       13.000                  12.000
recall_n                         0.812                   0.750
recall_usd                       0.839                   0.818
n_assigned                      23.000                  16.000
precision_n                      0.565                   0.750
precision_usd                    0.805                   0.838

manual review queue: 30 of 241 awards (12.4%); sample-harbor awards caught by auto or review: 16 of 16


## 7. BRIC/PDM subapplicants (early read for T2.5b / H3)

- **Matched:** all 441 FY2010+ subapplicants whose benefiting or ZIP county is a pilot county, against the crosswalk UEIs. OpenFEMA has no DUNS, and PDM competitive has no UEIs.
- **Hand-labelled (light review, user's choice):** only matched rows and harbor-keyword hits.

In [18]:
sam = pd.read_csv(OUT / "subapplicant_matches.csv", dtype={"award_id": str})
sp = sam[sam.version == cm.PRIMARY_VERSION]
t1 = sp.groupby("program_id").agg(subapplicants=("award_id", "size"), with_uei=("uei", lambda s: s.notna().sum()),
                                   auto_matched=("match_status", lambda s: (s == "auto").sum()),
                                   flagged=("match_status", lambda s: s.str.startswith("flagged").sum()))
t1.loc["all"] = t1.sum()
t1

,subapplicants,with_uei,auto_matched,flagged
program_id,,,,
bric,199,198,11,0
lpdm_legacy,7,0,0,0
pdm_competitive,206,0,0,0
pdm_cpf,29,11,0,0
all,441,209,11,0


In [19]:
by_v = sam.groupby(["version", "match_status"]).size().unstack(fill_value=0)
by_v

match_status,auto,none
version,,
own_op,10,431
own_op_nolow,10,431
own_op_parent,11,430
own_op_parent_nolow,10,431


In [20]:
rev = pd.read_csv(config.REFERENCE_DIR / "subapplicant_harbor_review_pilot.csv", dtype=str)
print(rev.groupby(["match_status", "label"]).size().to_string())
rev[["subapplication_id", "program_id", "fy", "recipient_name", "status", "label", "reason"]].assign(reason=lambda d: d.reason.str[:80])

match_status  label       
auto          not_harbor      9
              unclear         2
none          not_harbor      2
              other_harbor    1
              unclear         4


,subapplication_id,program_id,fy,recipient_name,status,label,reason
0,EMF-2020-BR-185-0001,bric,2020,TRINIDAD RANCHERIA,Subaward Accepted,not_harbor,"matched to CA-trinidad owner; project is hazard mitigation, not harbor: Trin..."
1,EMF-2020-BR-185-0002,bric,2020,TRINIDAD RANCHERIA,Subaward Accepted,not_harbor,"matched to CA-trinidad owner; project is hazard mitigation, not harbor: Trin..."
2,EMS-2020-BR-127-0001,bric,2020,QUILEUTE TRIBAL COUNCIL,Closeout Initiated,not_harbor,"matched to WA-la-push owner; project is hazard mitigation, not harbor: Quile..."
3,EMS-2020-BR-131-0004,bric,2020,COUNTY OF DOUGLAS,Ineligible,not_harbor,"matched to OR-winchester-bay owner; project is hazard mitigation, not harbor..."
4,EMF-2021-BR-092-0001,bric,2021,TRINIDAD RANCHERIA,Submitted to FEMA,not_harbor,"matched to CA-trinidad owner; project is hazard mitigation, not harbor: Arch..."
5,EMF-2022-BR-020-0001,bric,2022,TRINIDAD RANCHERIA,Submitted to FEMA,not_harbor,"matched to CA-trinidad owner; project is hazard mitigation, not harbor: Mana..."
6,EMF-2022-BR-020-0002,bric,2022,TRINIDAD RANCHERIA,Subaward Accepted,not_harbor,"matched to CA-trinidad owner; project is hazard mitigation, not harbor: Cher..."
7,EMF-2022-BR-020-0003,bric,2022,TRINIDAD RANCHERIA,Submitted to FEMA,not_harbor,"matched to CA-trinidad owner; project is hazard mitigation, not harbor: Slop..."
8,EMF-2022-BR-020-0004,bric,2022,TRINIDAD RANCHERIA,Submitted to FEMA,not_harbor,"matched to CA-trinidad owner; project is hazard mitigation, not harbor: Stab..."
9,EMF-2022-BR-044-0013,bric,2022,CITY OF RANCHO PALOS VERDES,Subaward Accepted,not_harbor,keyword hit is a place name; project: BRIC 2022-Rancho Palos Verdes Portugue...


**Read for T2.5b:**
- BRIC subapplicants link cleanly to owner entities by UEI (198 of 199 BRIC rows have one).
- In these 17 counties, the pilot owners' BRIC and PDM applications are not harbor projects: they are hazard plans, drought, slopes and seismic.
- The only harbor-entity applicant is the Port of Port Angeles (PDM FY2019, not approved), which is not a pilot harbor.
- So BRIC/PDM can measure whether a harbor *owner* applies (an H3 capacity signal), but not whether harbor *projects* apply.

## 8. Overrides applied (not part of the auto-match rate)

In [21]:
ov = pd.read_csv(cm.OVERRIDES_PATH, dtype=str, keep_default_na=False)
fin = cm.apply_overrides(prim, ov)
f = fin.merge(t_site[["award_id", "award_source", "label", "harbor_id"]], on=["award_id", "award_source"])
ok_final = (f.final_harbor_id.fillna("") == f.harbor_id.where(f.label == "sample_harbor", "")) | (f.label == "unclear")
print(ov.action.value_counts().to_string())
print(f"after overrides: {ok_final.sum()} of {len(f)} awards end on the truth harbor (or correctly on none); "
      f"remaining disagreements: {(~ok_final).sum()}")

action
unassign    10
assign       3
after overrides: 241 of 241 awards end on the truth harbor (or correctly on none); remaining disagreements: 0


## 9. Effort estimate for T2.1 / T2.2

- **Labelling time per award** (person estimate, by method; see the truth-set notes): about 0.5 min with record text alone, 2 min with an OpenFEMA title join, 8 min with a web lookup. The pilot averages **~0.85 min per award** (~202 min for 241).
- **Manual-review share:** under the workflow in section 6 (entity match + keyword auto-assigned; everything else matched or flagged goes to review), the queue is the share computed above.
- **Full-build scale:** the pilot's 241 awards are 241 of the 443 in-scope awards in all 42 counties (334 primes and 109 subawards). A full harbor-level build reviews the matched and keyword-flagged awards for all 58+ harbors, not every award.

In [22]:
p_all = pd.read_csv(cm.PILOT_DIR / "prime_awards.csv", dtype=str, low_memory=False)
s_all = pd.read_csv(cm.PILOT_DIR / "subawards.csv", dtype=str, low_memory=False)
inscope = (p_all.in_scope.eq("True") & p_all.in_fy_window.eq("True")).sum() + s_all.in_scope.eq("True").sum()
kwall = ((p_all.in_scope.eq("True") & p_all.in_fy_window.eq("True") & p_all.harbor_kw_flag.eq("True")).sum()
         + (s_all.in_scope.eq("True") & s_all.harbor_kw_flag.eq("True")).sum())
q = review.mean()
print(f"in-scope awards, all 42 counties (PIDP/EDA/BRIC/PDM pilot pull): {inscope}; keyword-flagged: {kwall}")
print(f"pilot review share {q:.1%} -> about {q*inscope:.0f} awards to review at 42-county scale for these programs, "
      f"~{q*inscope*2:.0f} person-minutes at 2 min each (matched/flagged awards are the harder ones)")

in-scope awards, all 42 counties (PIDP/EDA/BRIC/PDM pilot pull): 443; keyword-flagged: 58
pilot review share 12.4% -> about 55 awards to review at 42-county scale for these programs, ~110 person-minutes at 2 min each (matched/flagged awards are the harder ones)
